# HW2 · Kaggle №1: Data Understanding + Two-stage (Faster R-CNN) + One-stage Anchor-based (RetinaNet)

| § | Пункт рубрики | Бали |
|---|---|---|
| 1 | Data Understanding and Preprocessing (EDA, **лише в цьому ноутбуці**) | 1.00 |
| 2 | Two-stage Anchor-based: F0, sweep RPN, F1, F2, F3 (tuned), failure cases | 1.00 |
| 3 | One-stage Anchor-based: R0, R1, R2, R3, R4 (tuned) | 1.00 |

Решту (Anchor-free, Query-based, Your pipeline, сабмішени) робить ноутбук №2.

## Як це працює (однаково в обох ноутбуках)

**Два ноутбука = дві людини, різні моделі, без спільного сховища.** Кожен ноутбук пише все у свій `/kaggle/working/project`: `results/*.json` (метрики), `ckpt/` або `ckpt_tv/` (ваги), `data/` (фолди, графіки), `submissions/`. Ніяких Drive, rclone і секретів.

**Налаштування Kaggle:** *Add Input → Competitions →* `multiple-object-video-detection-hw-2-ucu-ai-2026` (спершу Join на сторінці змагання); Settings: **GPU (T4)**, **Internet = On** (ваги COCO качаються з інтернету); *Session options → Persistence → Files only* (якщо є в акаунті), щоб файли в `/kaggle/working` не зникали між сесіями.

**Крос-валідація.** Усі кліпи train+val об'єднані й розбиті на `N_FOLDS=3` фолди через `GroupKFold` **за кліпом** (кадри одного відео завжди в одному фолді, без витоку). Для фолду k модель вчиться на двох інших фолдах і оцінюється на фолді k (out-of-fold). Кожен експеримент іде для всіх фолдів: `results/<exp>_f<k>.json`; `summary()` дає **середнє по фолдах і std**. Однакові фолди, кадри, препроцесинг і функція `evaluate` для всіх моделей (фолди детерміновані, тож у двох ноутбуках однакові). Learned-препроцесинг (k-means anchors) рахується лише по train-частині фолду. Фінальна модель = **остання епоха** (best-за-val не обирається).

**Бюджет:** `STEP=4` (обґрунтування в EDA 1.3), `EPOCHS=8`, вхід 960 px, прогрів 1 епоха, `close_mosaic=2`. Приблизно 4.5k train-кадрів на фолд. Орієнтовно на T4: торчвіжн-модель ≈ 1 год на фолд, YOLO11s ≈ 15-40 хв, RT-DETR-L ≈ 1-1.5 год. **Виміряй першу епоху й за потреби змінюй `EPOCHS`/`STEP`/`RUN_FOLDS` (однаково в обох ноутбуках!).** Перша перевірка коду: `RUN_FOLDS=[0]`, `EPOCHS=1`, `FRACTION=0.02`.

**Обрив сесії (ліміт Kaggle ~9-12 год) і відновлення:** завершені експерименти пропускаються, а перервані тренування відновлюються з `last.pt` у `/kaggle/working/project`: лише якщо файли пережили сесію (Persistence = Files only або сесія жива). Інакше після обриву втрачається поточний фолд. Тому **перед закінченням сесії скачай `results_export.zip`** (остання клітинка) або натисни *Save Version*.

**Об'єднання результатів двох ноутбуків (для порівняння F/R з Y/D у звіті):** у кінці кожного ноутбука клітинка створює `results_export.zip` з `results/*.json`. Колега завантажує його як Kaggle Dataset, додає в Input свого ноутбука, розпаковує або вказує шлях у `EXTRA_RES` (клітинка «Об'єднання результатів»), і `summary()` показує обидва набори. Без цього кожен ноутбук бачить лише свої моделі, це теж працює.

## 0. Налаштування Kaggle

In [ ]:
!pip -q install pycocotools 
import os
from pathlib import Path
COMP = 'multiple-object-video-detection-hw-2-ucu-ai-2026'
ROOT = Path('/kaggle/working/project'); ROOT.mkdir(parents=True, exist_ok=True)       # УСІ результати цього ноутбука: results/, ckpt*/, data/, submissions/
WORK = Path('/kaggle/temp/work') if Path('/kaggle/temp').exists() else Path('/tmp/work'); WORK.mkdir(parents=True, exist_ok=True)
# Шукаємо папку змагання: <DATA>/train/annotations_coco.json, <DATA>/val/annotations_coco.json, <DATA>/test_images.csv (до 3 рівнів вкладеності)
hits = [h for pat in ('*', '*/*', '*/*/*') for h in Path('/kaggle/input').glob(pat + '/train/annotations_coco.json') if (h.parent.parent / 'test_images.csv').exists()]
if not hits:
    print('Вміст /kaggle/input:'); [print(' ', p) for p in sorted(Path('/kaggle/input').glob('*'))[:20]]
    raise FileNotFoundError("Не знайдено дані змагання. Правий сайдбар: Add Input -> Competitions -> '" + COMP + "' (спершу Join на сторінці змагання). "
                            "Очікувана структура: <папка>/train/annotations_coco.json, <папка>/val/annotations_coco.json, <папка>/test_images.csv. "
                            "Якщо ти додав ІНШИЙ датасет (напр. сирий VisDrone-VID з sequences/ та annotations/*.txt), цей ноутбук його не читає.")
DATA = hits[0].parent.parent
EXTRA_RES = []      # папки results/ колеги (див. розділ «Об'єднання результатів»), напр. [Path('/kaggle/input/<датасет-колеги>/project/results')]
print('DATA:', DATA, sorted(p.name for p in DATA.iterdir()), '| ROOT:', ROOT)

## 0.1 Протокол, дані, фолди, препроцесинг (спільне ядро, однакове в обох ноутбуках)

In [ ]:
import json, pickle, shutil, time, math, gc, contextlib, io, logging, functools, random, subprocess
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, matplotlib.patches as patches
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from sklearn.model_selection import GroupKFold
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

CACHE, RES, RUNS, SUBS, TVCK = (ROOT / n for n in ['data', 'results', 'ckpt', 'submissions', 'ckpt_tv'])
for d in (CACHE, RES, RUNS, SUBS, TVCK): d.mkdir(parents=True, exist_ok=True)

# ---- ПРОТОКОЛ (спільний для всіх моделей; міняти лише разом з усіма) ----
SEED, STEP   = 42, 4          # кожен 4-й кадр кліпу (обґрунтування: EDA 1.3, IoU сусідніх вибраних кадрів ~0.66)
IMGSZ        = 960            # довга сторона входу
SAVE_MAX     = 1280           # кадри зберігаємо зменшеними до 1280 px (швидше читання, менше диска)
EPOCHS       = 8              # бюджет епох для КОЖНОГО експерименту
CLOSE_MOSAIC = 2
WARMUP_EPOCHS, WORKERS = 1, 4   # прогрів 1 епоха (дефолт Ultralytics 3 з'їв би майже половину бюджету)
VAL_CONF, MAX_DET, NMS_IOU = 0.01, 500, 0.6
N_FOLDS      = 3              # крос-валідація за кліпами (GroupKFold)
RUN_FOLDS    = [0, 1, 2]      # які фолди запускати; для налагодження: [0]
FRACTION     = 1.0            # частка кадрів для швидкої перевірки коду (0.02)
INNER_VAL    = 500            # кадрів у «внутрішній» валідації (лише криві по епохах)
CLASSES = ['person', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'bus', 'motor']

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕМАЄ')

In [ ]:
# ───── Анотації COCO → таблиці. iscrowd=0: цільові об'єкти; iscrowd=1: ignored regions. cls = category_id - 1 ─────
OFF = {'train': 0, 'val': 10**7}            # зсув image_id, щоб id з train і val не перетиналися (пулимо train+val)

def load_split(split):
    coco = json.load(open(DATA / split / 'annotations_coco.json'))
    imgs = pd.DataFrame(coco['images']).rename(columns={'id': 'image_id'})
    imgs['path'] = [str(DATA / split / 'images' / str(v) / Path(f).name) for v, f in zip(imgs.video, imgs.file_name)]
    imgs['split'] = split; imgs['image_id'] += OFF[split]
    a = pd.DataFrame(coco['annotations']); a['image_id'] += OFF[split]
    a[['x', 'y', 'w', 'h']] = pd.DataFrame(a.bbox.tolist(), index=a.index)
    a['cls'] = a.category_id - 1; a['split'] = split
    a = a.merge(imgs[['image_id', 'video', 'frame_id', 'width', 'height']], on='image_id')
    return imgs, a[a.iscrowd == 0].drop(columns=['bbox']).reset_index(drop=True), a[a.iscrowd == 1].drop(columns=['bbox']).reset_index(drop=True)

T = {s: load_split(s) for s in OFF}      # без кешу: шляхи залежать від місця монтування даних
imgs_all = pd.concat([T[s][0] for s in OFF], ignore_index=True)       # усі кадри train+val
obj_all  = pd.concat([T[s][1] for s in OFF], ignore_index=True)
ign_all  = pd.concat([T[s][2] for s in OFF], ignore_index=True)
assert not set(T['train'][0].video) & set(T['val'][0].video), 'кліпи train/val перетинаються!'
OBJ_BY_IMG = {k: g for k, g in obj_all.groupby('image_id')}
IGN_BY_IMG = {k: g[['x', 'y', 'w', 'h']].values for k, g in ign_all.groupby('image_id')}

# ───── Фолди: ВСІ кадри одного кліпу в одному фолді (GroupKFold за video) ─────
fold_of = {}
for k, (_, te) in enumerate(GroupKFold(N_FOLDS).split(imgs_all, groups=imgs_all.video)):
    fold_of.update({v: k for v in imgs_all.video.iloc[te].unique()})
pd.Series(fold_of).to_csv(CACHE / 'folds.csv')

FR = imgs_all.sort_values(['video', 'frame_id'])
FR = FR[FR.groupby('video').cumcount() % STEP == 0].copy()           # підвибірка: кожен STEP-й кадр
FR['fold'] = FR.video.map(fold_of)
if FRACTION < 1: FR = FR.sample(frac=FRACTION, random_state=SEED)
FR = FR.reset_index(drop=True)

TRAIN = lambda k: FR[FR.fold != k]                                   # train-частина фолду k
VAL   = lambda k: FR[FR.fold == k]                                   # out-of-fold валідація (кліпи, яких модель не бачила)
VSUB  = lambda k: VAL(k).sample(min(INNER_VAL, len(VAL(k))), random_state=SEED)   # мала підвибірка для кривих по епохах
print(FR.groupby('fold').agg(clips=('video', 'nunique'), frames=('image_id', 'size')))

In [ ]:
def prep_all(root, mask):
    """Копія кадрів FR у root: картинка (<=SAVE_MAX px, ignored regions замальовані сірим якщо mask) + YOLO-мітка з наших таблиць."""
    root = Path(root)
    def one(r):
        img = root / 'images' / r.video / Path(r.path).name; lbl = root / 'labels' / r.video / (img.stem + '.txt')
        if img.exists(): return
        img.parent.mkdir(parents=True, exist_ok=True); lbl.parent.mkdir(parents=True, exist_ok=True)
        g = OBJ_BY_IMG.get(r.image_id); rows = []
        if g is not None:
            b = np.clip(np.c_[g.x, g.y, g.x + g.w, g.y + g.h], 0, [r.width, r.height] * 2); ok = (b[:, 2] > b[:, 0]) & (b[:, 3] > b[:, 1])
            rows = [f'{c} {(x1+x2)/2/r.width:.6f} {(y1+y2)/2/r.height:.6f} {(x2-x1)/r.width:.6f} {(y2-y1)/r.height:.6f}'
                    for c, (x1, y1, x2, y2) in zip(g.cls.values[ok], b[ok])]
        lbl.write_text('\n'.join(rows))
        im = Image.open(r.path).convert('RGB')
        if mask:
            a = np.array(im)
            for x, y, w, h in IGN_BY_IMG.get(r.image_id, []): a[int(y):int(np.ceil(y + h)), int(x):int(np.ceil(x + w))] = 114
            im = Image.fromarray(a)
        im.thumbnail((SAVE_MAX, SAVE_MAX)); im.save(img, quality=95)       # картинку пишемо останньою: вона маркер «готово»
    with ThreadPoolExecutor(8) as ex: list(ex.map(one, list(FR.itertuples())))
    print('кадрів підготовлено:', len(FR), '->', root)

def fold_yaml(k, root=None):
    """Списки кадрів для фолду k (train = інші фолди, val = INNER_VAL кадрів фолду k) + yaml для Ultralytics."""
    root = Path(root or WORK)
    for name, d in (('train', TRAIN(k)), ('val', VSUB(k))):
        (root / f'f{k}_{name}.txt').write_text('\n'.join(str(root / 'images' / r.video / Path(r.path).name) for r in d.itertuples()))
    y = root / f'data_f{k}.yaml'
    y.write_text(f"path: {root}\ntrain: {root}/f{k}_train.txt\nval: {root}/f{k}_val.txt\nnames: {dict(enumerate(CLASSES))}\n")
    return y

prep_all(WORK, mask=True)

In [ ]:
# ───── оцінка: ОДНА функція для всіх моделей (COCO mAP на кадрах фолду, ignored regions відкидаються) ─────
_GT = {}
def _gt():
    if not _GT:
        o = obj_all[obj_all.image_id.isin(FR.image_id)]
        g = dict(images=[dict(id=int(i), width=int(w), height=int(h)) for i, w, h in zip(FR.image_id, FR.width, FR.height)],
                 annotations=[dict(id=n + 1, image_id=int(i), category_id=int(c) + 1, bbox=[float(x), float(y), float(w), float(h)], area=float(w * h), iscrowd=0)
                              for n, (i, c, x, y, w, h) in enumerate(zip(o.image_id, o.cls, o.x, o.y, o.w, o.h))],
                 categories=[dict(id=i + 1, name=c) for i, c in enumerate(CLASSES)])
        cg = COCO(); cg.dataset = g
        with contextlib.redirect_stdout(io.StringIO()): cg.createIndex()
        _GT['coco'] = cg
    return _GT['coco']

def drop_in_ignore(xyxy, sc, cl, ign, thr=0.5):
    if len(xyxy) == 0 or len(ign) == 0: return xyxy, sc, cl
    gx1, gy1, gx2, gy2 = ign[:, 0], ign[:, 1], ign[:, 0] + ign[:, 2], ign[:, 1] + ign[:, 3]
    ix = (np.minimum(xyxy[:, None, 2], gx2) - np.maximum(xyxy[:, None, 0], gx1)).clip(0)
    iy = (np.minimum(xyxy[:, None, 3], gy2) - np.maximum(xyxy[:, None, 1], gy1)).clip(0)
    area = ((xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])).clip(1e-6)
    keep = (ix * iy).sum(1) / area < thr
    return xyxy[keep], sc[keep], cl[keep]

def coco_eval(preds, v, ms=float('nan')):
    """preds: {шлях: (xyxy, score, cls0)}; v: кадри. -> mAP, AP50, AP_small/medium/large, мс/кадр."""
    rows = []
    for iid, p in zip(v.image_id, v.path):
        xyxy, sc, cl = drop_in_ignore(*preds[str(Path(p))], IGN_BY_IMG.get(iid, np.zeros((0, 4))))
        if len(sc): rows.append(np.c_[np.full(len(sc), iid), xyxy[:, 0], xyxy[:, 1], xyxy[:, 2] - xyxy[:, 0], xyxy[:, 3] - xyxy[:, 1], sc, cl + 1])
    dets = np.concatenate(rows).astype(np.float64) if rows else np.zeros((0, 7))
    cg = _gt()
    with contextlib.redirect_stdout(io.StringIO()):
        E = COCOeval(cg, cg.loadRes(dets), 'bbox'); E.params.imgIds = [int(i) for i in v.image_id]
        E.params.maxDets = [1, 10, MAX_DET]; E.evaluate(); E.accumulate(); E.summarize()
    P = E.eval['precision']                  # [IoU, recall, клас, площа(all/small/medium/large), maxDets]; беремо maxDets=MAX_DET
    def ap(area=0, iou=None):
        p = P[:, :, :, area, 2] if iou is None else P[[int(np.argmin(abs(E.params.iouThrs - iou)))], :, :, area, 2]
        return float(p[p > -1].mean()) if (p > -1).any() else float('nan')
    return dict(mAP=ap(), AP50=ap(iou=0.5), AP_small=ap(1), AP_medium=ap(2), AP_large=ap(3), ms_per_img=ms)

def evaluate(pred_fn, v):
    preds, ms = pred_fn([str(p) for p in v.path])
    out = coco_eval(preds, v, ms); del preds; gc.collect(); return out

def make_submission(pred_fn, out_csv):
    test = pd.read_csv(DATA / 'test_images.csv'); part = Path(str(out_csv) + '.part'); done_ = {}
    if part.exists(): done_ = dict(pd.read_csv(part, dtype=str, keep_default_na=False).values)
    for clip, g in test[~test.image_id.isin(done_)].groupby('clip'):
        preds, _ = pred_fn([str(DATA / f) for f in g.file_name])
        for iid, f in zip(g.image_id, g.file_name):
            x, s, c = preds[str(DATA / f)]
            done_[iid] = ' '.join(f'{k} {p:.4f} {a:.1f} {b:.1f} {x2-a:.1f} {y2-b:.1f}' for (a, b, x2, y2), p, k in zip(x, s, c))
        pd.DataFrame(list(done_.items()), columns=['image_id', 'PredictionString']).to_csv(part, index=False)
    pd.DataFrame({'image_id': test.image_id, 'PredictionString': [done_[i] for i in test.image_id]}).to_csv(out_csv, index=False)
    print('збережено', out_csv, '| рядків:', len(test)); export_results()

# ───── облік експериментів: results/<exp>_f<k>.json  ─────
LOG_FILE = ROOT.parent / 'results_log.txt'      # /kaggle/working/results_log.txt: текстовий лог усіх метрик (сюди ж пише кожна епоха)
def log(msg):
    print(msg)
    with open(LOG_FILE, 'a') as f: f.write(msg + '\n')

def export_results():
    """Після КОЖНОГО експерименту оновлює архіви в /kaggle/working (панель Output): results_export.zip (метрики), submissions_export.zip (сабмішени)."""
    shutil.make_archive(str(ROOT.parent / 'results_export'), 'zip', ROOT, 'results')
    if (ROOT / 'submissions').exists() and any((ROOT / 'submissions').iterdir()): shutil.make_archive(str(ROOT.parent / 'submissions_export'), 'zip', ROOT, 'submissions')

def restore_from_log(text):
    """Відновлює results/*.json з рядків виду `назва_f<k> {'mAP': ...}` (results_log.txt або текст зі старих виводів). Вже наявні пропускає."""
    import ast, re; n = 0
    for line in text.splitlines():
        m = re.match(r"^\s*(\S+_f\d)\s+(\{.*\})\s*$", line)
        if m and not done(m.group(1)): json.dump({'metrics': ast.literal_eval(m.group(2)), 'cfg': {'restored_from_log': True}}, open(RES / f'{m.group(1)}.json', 'w'), indent=2); n += 1
    print('відновлено:', n)

def done(name): return (RES / f'{name}.json').exists()
def save_result(name, metrics, cfg, extra=None):
    json.dump({'metrics': {k: float(v) for k, v in metrics.items()}, 'cfg': cfg, **(extra or {})}, open(RES / f'{name}.json', 'w'), indent=2, default=str)
    log(f"{name} {({k: round(float(v), 4) for k, v in metrics.items()})}"); export_results()

def summary():
    """Середнє по фолдах (out-of-fold оцінка) + std mAP. Читає результати всіх трьох ноутбуків."""
    rows = []
    for f in sorted(p for d in [RES, *EXTRA_RES] for p in Path(d).glob('*_f[0-9].json')):
        e, k = f.stem.rsplit('_f', 1); rows.append({'exp': e, 'fold': int(k), **json.load(open(f))['metrics']})
    if not rows: return pd.DataFrame()
    g = pd.DataFrame(rows).groupby('exp'); out = g[['mAP', 'AP50', 'AP_small', 'AP_medium', 'AP_large', 'ms_per_img']].mean().round(4)
    out['mAP_std'] = g.mAP.std().round(4); out['n_folds'] = g.size(); return out

def better(exp, base, margin=0.002):
    """True, якщо exp дав більший mean mAP по фолдах, ніж base (з запасом на шум). Так обираємо складові «tuned» моделі."""
    s = summary(); return exp in s.index and base in s.index and s.loc[exp, 'mAP'] > s.loc[base, 'mAP'] + margin

### Відновлення метрик після скидання сесії (виконуй, лише якщо `results/` порожня)
Метрики кожного експерименту дописуються в `/kaggle/working/results_log.txt`; якщо Persistence увімкнено, файл переживе сесію й клітинка нижче відновить `results/*.json`. Також можна вставити в `PASTED` рядки виду `назва_f<k> {'mAP': ...}` зі старих виводів. Відновлені експерименти пропускаються при тренуванні, тож **не відновлюй те, чиї ваги потрібні далі** (F0, якщо будеш запускати RPN sweep та аналіз помилок; F3).

In [ ]:
PASTED = """
R1_retina_anchors_kmeans_f0 {'mAP': 0.0306, 'AP50': 0.0997, 'AP_small': 0.0185, 'AP_medium': 0.0338, 'AP_large': 0.0415, 'ms_per_img': 40.962}
R1_retina_anchors_kmeans_f1 {'mAP': 0.0467, 'AP50': 0.1407, 'AP_small': 0.0286, 'AP_medium': 0.0513, 'AP_large': 0.0586, 'ms_per_img': 41.0744}
R1_retina_anchors_kmeans_f2 {'mAP': 0.0413, 'AP50': 0.1304, 'AP_small': 0.0267, 'AP_medium': 0.0457, 'AP_large': 0.0645, 'ms_per_img': 40.9715}
R2_retina_focal_g1_f0 {'mAP': 0.1027, 'AP50': 0.196, 'AP_small': 0.0171, 'AP_medium': 0.085, 'AP_large': 0.2604, 'ms_per_img': 39.055}
"""
restore_from_log((LOG_FILE.read_text() if LOG_FILE.exists() else "") + "\n" + PASTED)
print(sorted(p.name for p in RES.glob("*.json")))

In [ ]:
import torchvision, torchvision.transforms.functional as TF
import torch.nn.functional as F
from sklearn.cluster import KMeans
from torchvision.models.detection import fasterrcnn_resnet50_fpn, retinanet_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.ops import box_iou, generalized_box_iou_loss

MIN_SIZE = int(IMGSZ * 9 / 16)

class DetDS(torch.utils.data.Dataset):
    """Кадри з підготовленої (masked) копії; боксі xyxy у пікселях масштабованого кадру; labels = cls+1 (0 = фон)."""
    def __init__(self, fr, flip=True): self.fr, self.flip = fr.reset_index(drop=True), flip
    def __len__(self): return len(self.fr)
    def __getitem__(self, i):
        r = self.fr.iloc[i]; s = IMGSZ / max(r.width, r.height)
        im = Image.open(WORK / 'images' / r.video / Path(r.path).name).convert('RGB').resize((round(r.width * s), round(r.height * s)), Image.BILINEAR)
        x = TF.to_tensor(im); g = OBJ_BY_IMG.get(r.image_id)
        if g is None: b, l = np.zeros((0, 4), np.float32), np.zeros(0, np.int64)
        else:
            b = np.c_[g.x, g.y, g.x + g.w, g.y + g.h].astype(np.float32) * s; l = (g.cls.values + 1).astype(np.int64)
            ok = ((b[:, 2] - b[:, 0]) > 1) & ((b[:, 3] - b[:, 1]) > 1); b, l = b[ok], l[ok]
        if self.flip and random.random() < 0.5:
            x = x.flip(-1); W = x.shape[-1]; b = b.copy(); b[:, [0, 2]] = W - b[:, [2, 0]]
        return x, {'boxes': torch.from_numpy(b), 'labels': torch.from_numpy(l)}

def tv_pred(model, imgsz, bs=4):
    @torch.no_grad()
    def fn(paths):
        model.eval(); out, t = {}, 0.
        for i in range(0, len(paths), bs):
            xs, ss = [], []
            for p in paths[i:i+bs]:
                im = Image.open(p).convert('RGB'); s = imgsz / max(im.size)
                xs.append(TF.to_tensor(im.resize((round(im.width * s), round(im.height * s)), Image.BILINEAR)).cuda()); ss.append(s)
            torch.cuda.synchronize(); t0 = time.time()
            with torch.autocast('cuda'): res = model(xs)
            torch.cuda.synchronize(); t += time.time() - t0
            for p, s, r in zip(paths[i:i+bs], ss, res):
                out[str(Path(p))] = (r['boxes'].float().cpu().numpy() / s, r['scores'].float().cpu().numpy(), (r['labels'] - 1).cpu().numpy())
        return out, 1000 * t / max(len(paths), 1)          # мс/кадр лише для forward моделі
    return fn

def train_tv(exp, k, build, cfg, lr=0.005, bs=4, clip=None):
    """Тренування torchvision-детектора на фолді k. Відновлюється з last.pt. Фінальна модель = остання епоха (без відбору за val)."""
    name = f'{exp}_f{k}'
    if done(name): print('Вже зроблено:', name); return
    ck = TVCK / name; ck.mkdir(exist_ok=True)
    model = build().cuda(); params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=1e-4)
    dl = torch.utils.data.DataLoader(DetDS(TRAIN(k)), batch_size=bs, shuffle=True, num_workers=WORKERS, collate_fn=lambda b: tuple(zip(*b)), drop_last=True,
        generator=torch.Generator().manual_seed(SEED + k), worker_init_fn=lambda w: random.seed(SEED + w))
    total, warm = EPOCHS * len(dl), min(500, len(dl)); scaler = torch.amp.GradScaler('cuda'); start, it, hist = 0, 0, []
    if (ck / 'last.pt').exists():
        s = torch.load(ck / 'last.pt', map_location='cuda', weights_only=False)
        model.load_state_dict(s['model']); opt.load_state_dict(s['opt']); start, it, hist = s['epoch'] + 1, s['it'], s['hist']; print('відновлено з епохи', start)
    for ep in range(start, EPOCHS):
        model.train(); t0 = time.time(); run, n = 0., 0
        for xs, tg in dl:
            xs = [x.cuda(non_blocking=True) for x in xs]; tg = [{a: b.cuda() for a, b in t.items()} for t in tg]
            f = min(1., (it + 1) / warm) * 0.5 * (1 + math.cos(math.pi * it / total))
            for g in opt.param_groups: g['lr'] = lr * f
            with torch.autocast('cuda'): loss = sum(model(xs, tg).values())
            if not torch.isfinite(loss): opt.zero_grad(set_to_none=True); it += 1; continue
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            if clip: scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, clip)
            scaler.step(opt); scaler.update(); it += 1; run += loss.item(); n += 1
        m = evaluate(tv_pred(model, IMGSZ), VSUB(k))             # крива по епохах (лише для графіка)
        hist.append(dict(epoch=ep + 1, loss=run / max(n, 1), sec=time.time() - t0, **m))
        log(f"{name} ep{ep+1}/{EPOCHS} loss {run/max(n,1):.3f} | val-sub mAP {m['mAP']:.4f} AP_small {m['AP_small']:.4f} | {time.time()-t0:.0f} с")
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'epoch': ep, 'it': it, 'hist': hist}, ck / 'last.pt')
    torch.save(model.state_dict(), ck / 'model.pt')
    save_result(name, evaluate(tv_pred(model, IMGSZ), VAL(k)), dict(cfg, fold=k), {'hist': hist})
    (ck / 'last.pt').unlink(missing_ok=True)      # чекпоінт з оптимізатором більше не потрібен (економимо диск)
    del model, opt; gc.collect(); torch.cuda.empty_cache()

def load_tv(exp, k, build):
    m = build().cuda(); m.load_state_dict(torch.load(TVCK / f'{exp}_f{k}' / 'model.pt', map_location='cuda')); return m.eval()

## 1. Data Understanding and Preprocessing (1.00)
⚪ **Запускати один раз.** Рубрика: *розподіл класів, розмірів, об'єктів на кадр і як це змінюється між послідовностями; які спостереження впливають на вибори; конвертація анотацій, ignored regions, нецільові класи; візуальна перевірка; сплит за послідовностями; крок підвибірки; resize, нормалізація, аугментації.*

**Обробка анотацій:** `iscrowd=0` цільові об'єкти (8 класів); `iscrowd=1` це **ignored regions** (натовпи, щільні парковки, VisDrone «others»). Клас переводиться в 0-based. У train ignored regions замальовуються сірим (114); при оцінці детекції ≥50 % площі в ignored regions відкидаються.

**Підвибірка:** кожен 4-й кадр кліпу (`STEP=4`): див. таблицю в 1.3; на відстані 4 кадри медіанний IoU того ж об'єкта ≈ 0.66 (сусідні кадри дублюють ~90 % інформації), тож крок 4 прибирає майже-дублікати, а бюджет GPU йде на більше епох і 3 фолди.

**Спліт:** за кліпами (GroupKFold, 3 фолди), кадри одного відео не потрапляють одночасно в train і val.

**Препроцесинг:** YOLO/RT-DETR: letterbox до `IMGSZ`, /255, стандартні аугментації Ultralytics (mosaic з `close_mosaic`, HSV, flip, scale, translate, erasing). torchvision: довга сторона до `IMGSZ`, ImageNet mean/std всередині моделі, лише horizontal flip (це різниця протоколу, треба згадати у звіті). `SEED=42`.

**Записати у звіт своїми словами:** частку small/medium/large, медіанний розмір, об'єктів на кадр (mean/p95/max), частку кадрів з ignored regions, різницю між кліпами, і які висновки з цього (який `imgsz`, чому дрібні anchors, чому NMS ризикований у щільних сценах).

In [ ]:
# 1.1 розподіли: класи, розміри, щільність, ignored regions (усі кадри train+val)
obj = obj_all.copy()
obj['size'] = np.sqrt(obj.w * obj.h); obj['rel'] = obj['size'] / np.sqrt(obj.width * obj.height)
obj['bucket'] = pd.cut(obj['size'], [0, 32, 96, 1e9], labels=['small', 'medium', 'large'])
fig, ax = plt.subplots(2, 3, figsize=(18, 9))
cc = obj.groupby(['split', 'cls']).size().unstack(0)
(cc / cc.sum()).set_axis(CLASSES).plot.bar(ax=ax[0, 0], title='Частка класів'); ax[0, 0].set_yscale('log')
ax[0, 1].hist(obj['size'], bins=np.logspace(0, 3, 60)); ax[0, 1].set_xscale('log')
for v in (32, 96): ax[0, 1].axvline(v, c='r', ls='--')
ax[0, 1].set_title('sqrt(w*h), px (межі COCO small/medium)')
obj.boxplot(column='size', by='cls', ax=ax[0, 2]); ax[0, 2].set_yscale('log'); ax[0, 2].set_xticklabels(CLASSES, rotation=45); ax[0, 2].set_title('Розмір по класах')
per_frame = obj.groupby(['split', 'video', 'frame_id']).size().rename('n').reset_index()
per_frame['video'] = pd.Categorical(per_frame.video, per_frame.groupby('video').n.mean().sort_values().index)
per_frame.boxplot(column='n', by='video', ax=ax[1, 0], rot=90, fontsize=6); ax[1, 0].set_title("Об'єктів на кадр по кліпах")
imgs_all.drop_duplicates('video').groupby(['width', 'height']).size().plot.bar(ax=ax[1, 1], title='Кліпів за роздільністю')
ax[1, 2].hist((ign_all.w * ign_all.h / (ign_all.width * ign_all.height)).groupby(ign_all.image_id).sum(), bins=40); ax[1, 2].set_title('Частка кадру в ignored regions')
plt.suptitle(''); plt.tight_layout(); plt.savefig(CACHE / 'eda.png', dpi=110); plt.show()
print(obj.groupby('split').bucket.value_counts(normalize=True).unstack().round(3))
print('медіана sqrt(area), px: %.1f | відн. розмір: %.4f' % (obj['size'].median(), obj['rel'].median()))
print("об'єктів/кадр: mean %.1f, p95 %d, max %d" % (per_frame.n.mean(), per_frame.n.quantile(.95), per_frame.n.max()))
print('кадрів з ignored regions: %.1f%%' % (100 * ign_all.image_id.nunique() / len(imgs_all)))

In [ ]:
# 1.2 відмінності між кліпами, склад фолдів, нецільові класи, k-means anchors (для звіту)
from sklearn.cluster import KMeans
f = FR.groupby('fold').agg(clips=('video', 'nunique'), frames=('image_id', 'size'))
f['objects'] = obj_all.merge(FR[['image_id', 'fold']], on='image_id').groupby('fold').size(); f['obj_per_frame'] = (f.objects / f.frames).round(1)
print('Склад фолдів (за кліпами, без перетину):'); print(f)

mix = pd.crosstab(obj_all.video, obj_all.cls, normalize='index'); mix.columns = CLASSES
fig, ax = plt.subplots(1, 2, figsize=(16, 7))
im = ax[0].imshow(mix.values, aspect='auto'); ax[0].set_xticks(range(8)); ax[0].set_xticklabels(CLASSES, rotation=45)
ax[0].set_title('Частка класів по кліпах'); plt.colorbar(im, ax=ax[0])
med = obj_all.assign(s=np.sqrt(obj_all.w * obj_all.h)).groupby('video').s.median().sort_values()
ax[1].bar(range(len(med)), med.values); ax[1].set_title("Медіанний розмір об'єкта по кліпах, px")
plt.tight_layout(); plt.savefig(CACHE / 'per_clip.png', dpi=110); plt.show()

if 'orig_category_id' in obj_all: print(pd.crosstab(obj_all.orig_category_id, obj_all.cls.map(dict(enumerate(CLASSES)))))
if 'orig_category_id' in ign_all: print('Ignored regions за orig_category_id:\n', ign_all.orig_category_id.value_counts().to_string())

sc = IMGSZ / np.maximum(obj_all.width, obj_all.height); wh = np.c_[obj_all.w * sc, obj_all.h * sc]
km = KMeans(9, n_init=3, random_state=SEED).fit(np.log(wh[::5]))
A9 = np.exp(km.cluster_centers_); A9 = A9[np.argsort(A9[:, 0] * A9[:, 1])]
print(f'k-means anchors (w,h) при imgsz={IMGSZ}:\n', A9.round(1)); print('h/w (p10/p50/p90):', np.percentile(wh[:, 1] / wh[:, 0], [10, 50, 90]).round(2))
print("частка об'єктів < 16 px при imgsz=%d: %.1f%%" % (IMGSZ, 100 * (np.sqrt(wh[:, 0] * wh[:, 1]) < 16).mean()))

In [ ]:
# 1.3 обґрунтування кроку підвибірки: медіанний IoU того ж об'єкта між кадрами на відстані k (усі кадри)
def track_iou(o, k):
    a = o[['video', 'track_id', 'frame_id', 'x', 'y', 'w', 'h']]
    m = a.merge(a.assign(frame_id=a.frame_id - k), on=['video', 'track_id', 'frame_id'], suffixes=('', '_b'))
    ix = (np.minimum(m.x + m.w, m.x_b + m.w_b) - np.maximum(m.x, m.x_b)).clip(0); iy = (np.minimum(m.y + m.h, m.y_b + m.h_b) - np.maximum(m.y, m.y_b)).clip(0)
    return (ix * iy / (m.w * m.h + m.w_b * m.h_b - ix * iy)).median()
o = obj_all[obj_all.track_id >= 0]
for k in [1, 2, 3, 4, 5, 10]: print(f"крок {k}: медіанний IoU = {track_iou(o, k):.3f}")

In [ ]:
# 1.4 візуальна перевірка конвертації: кольорові рамки = GT, червоні заливки = ignored regions
def show(n=6, seed=SEED):
    sample = FR.sample(n, random_state=seed)
    fig, axs = plt.subplots(2, n // 2, figsize=(7 * n // 2, 8))
    for ax_, r in zip(axs.ravel(), sample.itertuples()):
        ax_.imshow(Image.open(r.path)); ax_.axis('off'); ax_.set_title(f'{r.video}/{r.frame_id} ({r.width}x{r.height}) fold {r.fold}', fontsize=9)
        g = OBJ_BY_IMG.get(r.image_id)
        if g is not None:
            for b in g.itertuples(): ax_.add_patch(patches.Rectangle((b.x, b.y), b.w, b.h, fill=False, ec=plt.cm.tab10(int(b.cls)), lw=1))
        for x, y, w, h in IGN_BY_IMG.get(r.image_id, []): ax_.add_patch(patches.Rectangle((x, y), w, h, fill=True, fc='red', alpha=.3, ec='red'))
    plt.tight_layout(); plt.savefig(CACHE / 'gt_samples.png', dpi=100); plt.show()
show()

## 2. Two-stage Anchor-based: Faster R-CNN (1.00)
Рубрика: *претреновані ваги; задокументувати налаштування; ≥2 параметри (RPN proposals, RoI sampling, box loss, LR); mAP, AP50, AP_small, швидкість; внесок RPN і другої стадії; ≥3 failure cases.*

**Налаштування:** `fasterrcnn_resnet50_fpn`, ваги COCO, голова класифікатора на 9 виходів (8 + фон); SGD (momentum 0.9, wd 1e-4), lr 0.005 при batch 4, прогрів 500 ітерацій, косинус, AMP; вхід `IMGSZ`; flip.

| id | Що змінюємо |
|---|---|
| F0 | baseline |
| F0-sweep | `rpn_post_nms_top_n_test` ∈ {100, 300, 1000, 2000} (лише інференс) |
| F1 | RoI sampling: `box_batch_size_per_image=256`, `box_positive_fraction=0.5` |
| F2 | box loss: Smooth L1 → GIoU |
| F3 | **tuned**: комбінація складових F1/F2, що покращили baseline (вибір автоматичний за `summary()`, поріг +0.002 mAP) |

In [ ]:
def build_frcnn(num_classes=9, **kw):
    m = fasterrcnn_resnet50_fpn(weights='DEFAULT', min_size=MIN_SIZE, max_size=IMGSZ, box_score_thresh=VAL_CONF,
                                box_nms_thresh=NMS_IOU, box_detections_per_img=MAX_DET, **kw)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, num_classes)
    return m

FCFG = dict(model='fasterrcnn_resnet50_fpn', imgsz=IMGSZ, epochs=EPOCHS, lr=0.005, batch=4)
# F0: baseline
for k in RUN_FOLDS: train_tv('F0_frcnn_base', k, build_frcnn, FCFG)

In [ ]:
# F0-sweep: кількість RPN proposals на інференсі + recall пропозицій (внесок RPN)
@torch.no_grad()
def rpn_recall(model, v, ns=(100, 300, 1000, 2000), n_img=200, thr=0.5):
    model.eval(); model.rpn._pre_nms_top_n['testing'] = 2000; model.rpn._post_nms_top_n['testing'] = max(ns)
    hit = {n: 0 for n in ns}; hit_s = {n: 0 for n in ns}; tot = tot_s = 0
    for r in v.sample(min(n_img, len(v)), random_state=SEED).itertuples():
        g = OBJ_BY_IMG.get(r.image_id)
        if g is None: continue
        im = Image.open(r.path).convert('RGB'); s = IMGSZ / max(im.size)
        x = TF.to_tensor(im.resize((round(im.width * s), round(im.height * s)), Image.BILINEAR)).cuda()
        il, _ = model.transform([x]); props, _ = model.rpn(il, model.backbone(il.tensors)); P = props[0]
        fy, fx = il.image_sizes[0][0] / x.shape[1], il.image_sizes[0][1] / x.shape[2]
        G = torch.tensor(np.c_[g.x, g.y, g.x + g.w, g.y + g.h] * s, dtype=torch.float32).cuda(); G[:, [0, 2]] *= fx; G[:, [1, 3]] *= fy
        small = torch.tensor(np.sqrt(g.w.values * g.h.values) < 32).cuda(); tot += len(G); tot_s += int(small.sum())
        if P.shape[0] == 0: continue
        iou = box_iou(G, P)
        for n in ns:
            ok = iou[:, :n].max(1).values >= thr; hit[n] += int(ok.sum()); hit_s[n] += int((ok & small).sum())
    model.rpn._post_nms_top_n['testing'] = 1000
    return pd.DataFrame({'recall@IoU0.5': {n: hit[n] / tot for n in ns}, 'recall_small': {n: hit_s[n] / max(tot_s, 1) for n in ns}})

for k in RUN_FOLDS:
    fr = load_tv('F0_frcnn_base', k, build_frcnn); print(f'── фолд {k}: recall пропозицій'); print(rpn_recall(fr, VAL(k)).round(3))
    rows = {}
    for n in (100, 300, 1000, 2000):
        fr.rpn._pre_nms_top_n['testing'] = max(1000, n); fr.rpn._post_nms_top_n['testing'] = n; rows[n] = evaluate(tv_pred(fr, IMGSZ), VSUB(k))
    sweep = pd.DataFrame(rows).T.round(4); sweep.index.name = 'rpn_post_nms_top_n_test'; print(sweep); sweep.to_csv(RES / f'F0_rpn_proposals_sweep_f{k}.csv')
    del fr; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# F1: RoI sampling
for k in RUN_FOLDS:
    train_tv('F1_frcnn_roi256_pos05', k, lambda: build_frcnn(box_batch_size_per_image=256, box_positive_fraction=0.5),
             dict(FCFG, box_batch_size_per_image=256, box_positive_fraction=0.5))

In [ ]:
# F2: Smooth L1 -> GIoU для регресії боксів у RoI head.
# Дельти RoI head декодуємо відносно одиничного боксу; IoU/GIoU інваріантні до масштабування осей, тож результат збігається з розрахунком у системі координат пропозиції.
import torchvision.models.detection.roi_heads as _rh
_orig_frcnn_loss = _rh.fastrcnn_loss
GIOU_W = 10.0       # вага втрати: власний вибір (за аналогією з конфігами MMDetection); обґрунтувати або підібрати

def _decode_unit(d):
    cx, cy = 0.5 + d[:, 0] / 10., 0.5 + d[:, 1] / 10.; clampv = math.log(1000. / 16)
    w, h = torch.exp((d[:, 2] / 5.).clamp(max=clampv)), torch.exp((d[:, 3] / 5.).clamp(max=clampv))
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], 1)

def fastrcnn_loss_giou(class_logits, box_regression, labels, regression_targets):
    labels = torch.cat(labels, 0); regression_targets = torch.cat(regression_targets, 0)
    cls_loss = F.cross_entropy(class_logits, labels)
    pos = torch.where(labels > 0)[0]; N = class_logits.shape[0]
    pred = box_regression.reshape(N, -1, 4)[pos, labels[pos]].float()
    box_loss = generalized_box_iou_loss(_decode_unit(pred), _decode_unit(regression_targets[pos].float()), reduction='sum') / labels.numel()
    return cls_loss, box_loss * GIOU_W

@contextlib.contextmanager
def giou_patch(on=True):
    if on: _rh.fastrcnn_loss = fastrcnn_loss_giou
    try: yield
    finally: _rh.fastrcnn_loss = _orig_frcnn_loss          # завжди повертаємо стандартну втрату

with giou_patch(True):
    for k in RUN_FOLDS: train_tv('F2_frcnn_giou', k, build_frcnn, dict(FCFG, box_loss='GIoU', giou_weight=GIOU_W))

In [ ]:
# F3: tuned = складові F1/F2, що дали приріст mAP над F0 (вибір за out-of-fold, поріг +0.002)
assert 'F0_frcnn_base' in summary().index, 'Немає F0: порахуй його (або restore_from_log) перед F3'
use_roi  = better('F1_frcnn_roi256_pos05', 'F0_frcnn_base')
use_giou = better('F2_frcnn_giou', 'F0_frcnn_base')
print('у tuned входять: RoI sampling =', use_roi, '| GIoU =', use_giou)
roi_kw = dict(box_batch_size_per_image=256, box_positive_fraction=0.5) if use_roi else {}
if use_roi or use_giou:
    FT, FT_BUILD = 'F3_frcnn_tuned', (lambda: build_frcnn(**roi_kw))
    with giou_patch(use_giou):
        for k in RUN_FOLDS: train_tv(FT, k, FT_BUILD, dict(FCFG, giou=use_giou, **roi_kw))
else:
    FT, FT_BUILD = 'F0_frcnn_base', build_frcnn; print('жоден варіант не покращив baseline: tuned = baseline (це теж результат для звіту)')

### Аналіз failure cases (≥3) і внесок RPN / другої стадії
Аналізується **tuned** Faster R-CNN (F3, або F0 якщо tuned = baseline), фолд `RUN_FOLDS[0]`, кадри `VSUB`. Зелені рамки GT, блакитні правильні детекції, червоні хибні, жовті пропущені GT (поріг score 0.4).

**Записати у звіт своїми словами:** для кожного з 3 кадрів що не так і чому (дрібний розмір, перекриття, схожі класи van/car/truck, ignored regions, поріг score). Внесок RPN: recall пропозицій при 100/300/1000/2000 проти AP50 фінальної моделі: якщо recall близький до 1, а AP50 значно нижчий, обмежує друга стадія; якщо recall для small низький, обмежує RPN. Внесок другої стадії: уточнення боксів і класифікація (розрив AP50 vs mAP показує якість локалізації).

In [ ]:
def match_frame(preds, r, thr=0.4, iou_thr=0.5):
    x, s, c = preds[str(Path(r.path))]; kk = s >= thr; x, s, c = x[kk], s[kk], c[kk]
    x, s, c = drop_in_ignore(x, s, c, IGN_BY_IMG.get(r.image_id, np.zeros((0, 4))))
    g = OBJ_BY_IMG.get(r.image_id)
    G = np.c_[g.x, g.y, g.x + g.w, g.y + g.h] if g is not None else np.zeros((0, 4)); gc_ = g.cls.values if g is not None else np.zeros(0, int)
    used = np.zeros(len(G), bool); tp, fp = [], []
    iou = box_iou(torch.tensor(x, dtype=torch.float32), torch.tensor(G, dtype=torch.float32)).numpy() if len(x) and len(G) else np.zeros((len(x), len(G)))
    for j in np.argsort(-s):
        cand = [i for i in np.where(~used & (gc_ == c[j]))[0] if iou[j, i] >= iou_thr] if len(G) else []
        if cand: i = max(cand, key=lambda q: iou[j, q]); used[i] = True; tp.append(j)
        else: fp.append(j)
    return x, G, tp, fp, np.where(~used)[0]

def frame_errors(preds, v):
    rows = []
    for r in v.itertuples():
        x, G, tp, fp, fn = match_frame(preds, r)
        small_fn = int((np.sqrt((G[fn, 2] - G[fn, 0]) * (G[fn, 3] - G[fn, 1])) < 32).sum()) if len(fn) else 0
        rows.append(dict(image_id=r.image_id, n_gt=len(G), tp=len(tp), fp=len(fp), fn=len(fn), small_fn=small_fn))
    return pd.DataFrame(rows)

def show_cases(preds, cases, fname):
    fig, axs = plt.subplots(1, len(cases), figsize=(8 * len(cases), 6))
    for ax_, (title, r) in zip(np.atleast_1d(axs), cases):
        x, G, tp, fp, fn = match_frame(preds, r); ax_.imshow(Image.open(r.path)); ax_.axis('off')
        for i, gb in enumerate(G): ax_.add_patch(patches.Rectangle((gb[0], gb[1]), gb[2] - gb[0], gb[3] - gb[1], fill=False, ec='yellow' if i in fn else 'lime', lw=1.2))
        for j in tp: ax_.add_patch(patches.Rectangle((x[j][0], x[j][1]), x[j][2] - x[j][0], x[j][3] - x[j][1], fill=False, ec='cyan', lw=1))
        for j in fp: ax_.add_patch(patches.Rectangle((x[j][0], x[j][1]), x[j][2] - x[j][0], x[j][3] - x[j][1], fill=False, ec='red', lw=1.2))
        ax_.set_title(f'{title}\n{r.video}/{r.frame_id}: GT {len(G)}, TP {len(tp)}, FP {len(fp)}, FN {len(fn)}', fontsize=10)
    plt.tight_layout(); plt.savefig(CACHE / fname, dpi=100); plt.show()

k0 = RUN_FOLDS[0]; v = VSUB(k0); fr = load_tv(FT, k0, FT_BUILD)
preds_fr, _ = tv_pred(fr, IMGSZ)([str(p) for p in v.path])
err = frame_errors(preds_fr, v); err['fn_rate'] = err.fn / err.n_gt.clip(1); dense = err[err.n_gt >= 60]
row = lambda iid: next(v[v.image_id == iid].itertuples())
cases = [('1) найбільше пропусків у щільній сцені', row((dense if len(dense) else err).sort_values('fn_rate').image_id.iloc[-1])),
         ('2) найбільше хибних детекцій', row(err.sort_values('fp').image_id.iloc[-1])),
         ('3) найбільше пропущених дрібних', row(err.sort_values('small_fn').image_id.iloc[-1]))]
show_cases(preds_fr, cases, 'frcnn_failures.png'); print(err[['n_gt', 'tp', 'fp', 'fn', 'small_fn']].sum())
del fr, preds_fr; gc.collect(); torch.cuda.empty_cache()

## 3. One-stage Anchor-based: RetinaNet (1.00)
Рубрика: *той самий протокол; тюнінг anchors / focal loss / IoU-порогів; anchors за розмірами об'єктів з EDA; чи створює відсутність proposal-стадії проблеми і як їх вирішують; порівняти початкову й тюновану модель з тюнованим двостадійним.*

**Налаштування:** `retinanet_resnet50_fpn`, ваги COCO, голова класифікації на 9 виходів, SGD lr 0.0025 при batch 4, clip-grad 10, AMP, ті самі дані й вхід, що в Faster R-CNN. **K-means anchors рахуються лише по train-частині фолду.**

| id | Що змінюємо | Обґрунтування |
|---|---|---|
| R0 | baseline: anchors torchvision (32…512), focal α=0.25 γ=2, IoU fg 0.5 / bg 0.4 | |
| R1 | anchors з **k-means** (5 центрів за `sqrt(w·h)`, 3 масштаби на рівень, h/w = p10/p50/p90) | перша спроба «anchors за даними» |
| R1b | **default anchors × α**: всі розміри 32…512 множимо на α ∈ {0.25, 0.35, 0.5, 0.7, 1.0}; α обирається за macro-покриттям GT (small/medium/large рівноправні) на train-частині фолду | розміри прив'язані до stride; k-means за кількістю об'єктів з'їдає великі об'єкти (див. покриття й AP_large) |
| R2 | **focal loss**: γ=1.0 | менш агресивне заглушення легких прикладів |
| R3 | **IoU-пороги**: fg 0.4 / bg 0.3 | дрібні об'єкти рідко мають anchor з IoU ≥ 0.5 |
| R4 | **tuned**: комбінація тих із R1b, R2, R3, що покращили R0 (автоматично, поріг +0.002 mAP) | порівняння tuned RetinaNet із tuned Faster R-CNN |

**Записати у звіт:** чи створює відсутність proposal-стадії проблему (так: дисбаланс фону й об'єктів, десятки тисяч anchors, майже всі фон; рішення: focal loss, ATSS-подібне призначення, hard negative mining); порівняння R з F за mAP, AP_small і швидкістю.

In [ ]:
import torchvision.models.detection.retinanet as _rn
_orig_focal = _rn.sigmoid_focal_loss
def set_focal(alpha=0.25, gamma=2.0): _rn.sigmoid_focal_loss = functools.partial(_orig_focal, alpha=alpha, gamma=gamma)

def data_anchors(k, levels=5):
    """Anchors за розмірами об'єктів train-частини фолду k (learned preprocessing не бачить val-кліпів)."""
    t = obj_all[obj_all.image_id.isin(TRAIN(k).image_id)]
    s = np.sqrt(t.w.values * t.h.values) * IMGSZ / np.maximum(t.width, t.height).values
    c = np.sort(np.exp(KMeans(levels, n_init=3, random_state=SEED).fit(np.log(s[::3]).reshape(-1, 1)).cluster_centers_.ravel()))
    sizes = tuple((float(x / 2 ** (1/3)), float(x), float(x * 2 ** (1/3))) for x in c)
    q = np.percentile((t.h / t.w).values, [10, 50, 90]); ratios = tuple(tuple(float(v) for v in q) for _ in c)
    return sizes, ratios

def build_retina(anchors=None, **kw):
    extra = dict(anchor_generator=AnchorGenerator(sizes=anchors[0], aspect_ratios=anchors[1])) if anchors else {}
    m = retinanet_resnet50_fpn(weights='DEFAULT', min_size=MIN_SIZE, max_size=IMGSZ, score_thresh=VAL_CONF, nms_thresh=NMS_IOU, detections_per_img=MAX_DET, **extra, **kw)
    m.head.classification_head = RetinaNetClassificationHead(256, m.head.classification_head.num_anchors, 9)
    return m

# ───── покриття GT anchor-ами: яка частка GT має anchor з IoU>=0.5 (за COCO-групами розміру); рахується по train-частині фолду ─────
DEF_SIZES  = tuple((x, x * 2 ** (1/3), x * 2 ** (2/3)) for x in (32, 64, 128, 256, 512)); DEF_RATIOS = ((0.5, 1.0, 2.0),) * 5   # anchors torchvision за замовчуванням
ALPHAS = (0.25, 0.35, 0.5, 0.7, 1.0)

def scaled_anchors(alpha): return tuple(tuple(v * alpha for v in lvl) for lvl in DEF_SIZES), DEF_RATIOS    # розміри прив'язані до stride, масштабуємо всі рівні разом

def anchor_coverage(sizes, ratios, k, thr=0.5, n=30000):
    """Частка GT з best-IoU>=thr серед anchors з УРАХУВАННЯМ СІТКИ: anchor стоїть лише у вузлах з кроком stride (8,16,...,128), центр GT випадково
    зсунутий відносно сітки. Anchor, менший за stride, не може надійно накрити об'єкт, тож розмір anchor має відповідати stride свого рівня."""
    t = obj_all[obj_all.image_id.isin(TRAIN(k).image_id)]; t = t.iloc[np.random.RandomState(SEED).choice(len(t), min(n, len(t)), replace=False)]
    s = IMGSZ / np.maximum(t.width, t.height).values; gw, gh, so = t.w.values * s, t.h.values * s, np.sqrt(t.w.values * t.h.values)   # so: розмір в оригінальних px (як у COCO)
    u, v = np.random.RandomState(SEED + 1).uniform(-.5, .5, (2, len(gw))); best = np.zeros(len(gw))
    for lvl, (ls, lr) in enumerate(zip(sizes, ratios)):
        dx, dy = u * 8 * 2 ** lvl, v * 8 * 2 ** lvl                       # зсув найближчого вузла сітки відносно центра GT
        for sz in ls:
            for r in lr:                                                  # r = h/w, як у torchvision
                aw, ah = sz / np.sqrt(r), sz * np.sqrt(r)
                ix = (np.minimum(gw / 2, dx + aw / 2) - np.maximum(-gw / 2, dx - aw / 2)).clip(0); iy = (np.minimum(gh / 2, dy + ah / 2) - np.maximum(-gh / 2, dy - ah / 2)).clip(0)
                best = np.maximum(best, ix * iy / (gw * gh + aw * ah - ix * iy))
    cov = {nm: float((best[m] >= thr).mean()) for nm, m in [('small', so < 32), ('medium', (so >= 32) & (so < 96)), ('large', so >= 96)]}
    cov['macro'] = float(np.mean(list(cov.values()))); return cov

def choose_alpha(k):
    """Масштаб α для anchors за замовчуванням: максимум macro-покриття (три групи розміру рівноправні, щоб дрібні об'єкти не 'з'їли' великі)."""
    return max(ALPHAS, key=lambda a: anchor_coverage(*scaled_anchors(a), k)['macro'])

RCFG = dict(model='retinanet_resnet50_fpn', imgsz=IMGSZ, epochs=EPOCHS, lr=0.0025, batch=4)
A0 = data_anchors(RUN_FOLDS[0]); print('розміри anchors по рівнях FPN (P3…P7):', [tuple(round(v) for v in s) for s in A0[0]], '| h/w:', [round(v, 2) for v in A0[1][0]])

In [ ]:
# Покриття GT anchor-ами з урахуванням сітки (частка GT з best-IoU>=0.5): default / k-means (R1) / масштабовані default. Порівняй колонку large з AP_large у R1 і R2.
k0 = RUN_FOLDS[0]; rows = {'default (R0)': (DEF_SIZES, DEF_RATIOS), 'k-means (R1)': data_anchors(k0)}; rows.update({f'default x{a}': scaled_anchors(a) for a in ALPHAS})
print(pd.DataFrame({n: anchor_coverage(*v, k0) for n, v in rows.items()}).T.round(3)); print('вибраний α для фолду', k0, ':', choose_alpha(k0))

In [ ]:
# R0: baseline
set_focal(0.25, 2.0)
for k in RUN_FOLDS: train_tv('R0_retina_base', k, build_retina, RCFG, lr=0.0025, clip=10)

In [ ]:
# R1: anchors з k-means за кількістю об'єктів (свої для кожного фолду)
set_focal(0.25, 2.0)
for k in RUN_FOLDS:
    A = data_anchors(k); train_tv('R1_retina_anchors_kmeans', k, lambda: build_retina(anchors=A), dict(RCFG, anchors=str(A)), lr=0.0025, clip=10)

In [ ]:
# R1b: anchors за замовчуванням, масштабовані на α, обраний за покриттям GT (train-частина фолду; своя α для кожного фолду)
set_focal(0.25, 2.0)
for k in RUN_FOLDS:
    a = choose_alpha(k); A = scaled_anchors(a); print(f'фолд {k}: α = {a}')
    train_tv('R1b_retina_anchors_scaled', k, lambda: build_retina(anchors=A), dict(RCFG, anchors=str(A), alpha=a), lr=0.0025, clip=10)

In [ ]:
# R2: focal loss γ=1.0
set_focal(0.25, 1.0)
try:
    for k in RUN_FOLDS: train_tv('R2_retina_focal_g1', k, build_retina, dict(RCFG, focal_alpha=0.25, focal_gamma=1.0), lr=0.0025, clip=10)
finally: set_focal(0.25, 2.0)

In [ ]:
# R3: IoU-пороги призначення
set_focal(0.25, 2.0)
for k in RUN_FOLDS: train_tv('R3_retina_iou_fg04_bg03', k, lambda: build_retina(fg_iou_thresh=0.4, bg_iou_thresh=0.3), dict(RCFG, fg_iou_thresh=0.4, bg_iou_thresh=0.3), lr=0.0025, clip=10)

In [ ]:
# R4: tuned = складові R1b/R2/R3, що дали приріст mAP над R0 (out-of-fold, поріг +0.002)
need = ['R0_retina_base', 'R1b_retina_anchors_scaled', 'R2_retina_focal_g1', 'R3_retina_iou_fg04_bg03']
assert all(e in summary().index for e in need), f'бракує: {[e for e in need if e not in summary().index]}'
cand = {'R1': 'R1b_retina_anchors_scaled', 'R2': 'R2_retina_focal_g1', 'R3': 'R3_retina_iou_fg04_bg03'}
u = {c: better(e, 'R0_retina_base') for c, e in cand.items()}; print('у tuned входять:', u)
gamma = 1.0 if u['R2'] else 2.0; iou_kw = dict(fg_iou_thresh=0.4, bg_iou_thresh=0.3) if u['R3'] else {}
if any(u.values()):
    set_focal(0.25, gamma)
    try:
        for k in RUN_FOLDS:
            A = scaled_anchors(choose_alpha(k)) if u['R1'] else None
            train_tv('R4_retina_tuned', k, lambda: build_retina(anchors=A, **iou_kw), dict(RCFG, anchors=u['R1'], focal_gamma=gamma, **iou_kw), lr=0.0025, clip=10)
    finally: set_focal(0.25, 2.0)
else: print('жоден варіант не покращив R0: tuned = baseline')

## 4. Порівняння і зведена таблиця

In [ ]:
# Порівняння: initial vs tuned для двостадійного й одностадійного anchor-based (+ anchor-free/query-based, якщо додано results колеги через EXTRA_RES)
tab = summary()
order = ['F0_frcnn_base', 'F1_frcnn_roi256_pos05', 'F2_frcnn_giou', 'F3_frcnn_tuned', 'R0_retina_base', 'R1_retina_anchors_kmeans', 'R1b_retina_anchors_scaled', 'R2_retina_focal_g1', 'R3_retina_iou_fg04_bg03', 'R4_retina_tuned',
         'Y0_yolo11s_960', 'Y3_yolo11s_tuned', 'D0_rtdetr_l_960']
print(tab.loc[[e for e in order if e in tab.index]].to_string())      # mAP, AP50, AP_small/medium/large (розбивка за розміром), мс/кадр, std по фолдах

### Експорт і об'єднання результатів
Запусти клітинку нижче наприкінці роботи (і перед завершенням сесії): вона пакує `results/` (і `submissions/`, якщо є) у `results_export.zip` в `/kaggle/working`, звідки його можна скачати. Результати колеги підключаються так: завантаж його `results_export.zip` як Kaggle Dataset, додай у Input і виконай `EXTRA_RES = [Path('/kaggle/input/<датасет>/project/results')]` (або розпакуй zip і вкажи папку з `*.json`), потім `summary()`.

In [ ]:
shutil.make_archive('/kaggle/working/results_export', 'zip', ROOT, 'results')
if (ROOT / 'submissions').exists() and any((ROOT / 'submissions').iterdir()): shutil.make_archive('/kaggle/working/submissions_export', 'zip', ROOT, 'submissions')
print([p.name for p in Path('/kaggle/working').glob('*.zip')])

### Відтворення (для розділу звіту «Final model and run instructions»)
Залежності: `torch`/`torchvision` з образу Kaggle, `pycocotools`, `scikit-learn`; сід `SEED=42` (python/numpy/torch + генератор DataLoader); фолди зберігаються в `data/folds.csv`. Порядок запуску: налаштування → ядро → TV-блок → §1 EDA (один раз) → §2 → §3 → порівняння.

### Чекліст звіту для цього ноутбука
- [ ] **EDA:** частки small/medium/large, медіанний розмір, об'єктів/кадр, ignored regions, різниця між кліпами, склад фолдів; висновки для `imgsz`/anchors/NMS; обробка ignored/нецільових класів; крок підвибірки (таблиця IoU); сплит за кліпами; resize/нормалізація/аугментації.
- [ ] **Two-stage:** налаштування; ≥2 параметри (sweep proposals, RoI sampling, GIoU); mAP/AP50/AP_small/мс; внесок RPN (recall) і RoI-голови; ≥3 failure cases з причинами.
- [ ] **One-stage anchor-based:** anchors з k-means; focal; IoU-пороги; проблема дисбалансу без proposals і як її розв'язують; порівняння initial/tuned з tuned Faster R-CNN.
- [ ] Джерела: Faster R-CNN, FPN, RetinaNet/Focal Loss, ATSS, GIoU, torchvision. Невдалі експерименти теж у звіт.